In [0]:
%sql
-- Check whether Databricks can list the article files written by Lambda.
LIST 's3://databricks-news-articles/kontrast.at/' LIMIT 3;

In [0]:
-- Read one article and preview the first 500 characters of its HTML.
SELECT
    path,
    length AS size_bytes,
    substring(decode(content, 'UTF-8'), 1, 500) AS html_preview
FROM read_files(
    's3://databricks-news-articles/kontrast.at/1-mai-wehren-wir-uns-gegen-den-klassenkampf-von-oben.html',
    format => 'binaryFile'
);

In [0]:
%python
# Parse one archived article and show its source path with extracted fields.
from parse_article import parse_article

article_path = 's3://databricks-news-articles/kontrast.at/1-mai-wehren-wir-uns-gegen-den-klassenkampf-von-oben.html'
row = spark.read.format('binaryFile').load(article_path).select('path', 'content').first()
article = parse_article(row.content.decode('utf-8', errors='replace'))

print('SOURCE FILE:', row.path)
for field, value in article.items():
    print(f'{field.upper()}: {value}')

In [0]:
%python
# Build one typed Silver candidate from the parsed article and S3 path.
from datetime import datetime
from pyspark.sql.types import ArrayType, StringType, StructField, StructType, TimestampType

author_schema = StructType([StructField('name', StringType()), StructField('profile_url', StringType())])
link_schema = StructType([StructField('text', StringType()), StructField('url', StringType())])
schema = StructType([
    StructField('canonical_url', StringType()),
    StructField('headline', StringType()),
    StructField('body_text', StringType()),
    StructField('published_at', TimestampType()),
    StructField('modified_at', TimestampType()),
    StructField('authors', ArrayType(author_schema)),
    StructField('tags', ArrayType(StringType())),
    StructField('categories', ArrayType(StringType())),
    StructField('links', ArrayType(link_schema)),
    StructField('source_path', StringType()),
])

# Convert publisher date strings to timestamps and include the original file path.
record = dict(article)
record['published_at'] = datetime.fromisoformat(article['published_at'])
record['modified_at'] = datetime.fromisoformat(article['modified_at'])
record['source_path'] = row.path
silver_candidate = spark.createDataFrame([record], schema=schema)
silver_candidate.printSchema()
display(silver_candidate)